# PlantCLEF 2026 — soumission

Notebook de soumission pour Kaggle. La structure est celle de `submission-method.md` : les cellules 1 à 4, 6 et 7 sont fixes. Seule `predict_species` (cellule 5) change d'un run à l'autre.

**Ce run** retient les espèces dont la probabilité softmax, calculée sur l'image entière redimensionnée à la taille d'entrée du modèle, est au moins `0.1`. Les identifiants sont renvoyés par score décroissant. Si aucune espèce ne passe le seuil, la liste reste vide.

In [ ]:
import os, csv
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from tqdm import tqdm


In [ ]:
C = "/kaggle/input/competitions/plantclef-2026"
TEST_CSV    = f"{C}/PlantCLEF2025_test.csv"
SPECIES_CSV = f"{C}/species_ids.csv"
IMG_DIR     = f"{C}/PlantCLEF2025_test_images/PlantCLEF2025_test_images"
MODEL_CKPT  = "/kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar"

ID_COL  = "quadrat_id"   # vérifié : colonnes du CSV de test = quadrat_id, author, date, license
IMG_EXT = ".jpg"         # vérifié : les 2 105 images sont en .jpg, noms = quadrat_id + extension

def image_path(quadrat_id: str) -> str:
    return f"{IMG_DIR}/{quadrat_id}{IMG_EXT}"

print("GPU :", torch.cuda.is_available())


In [ ]:
df_species = pd.read_csv(SPECIES_CSV)
class_map = df_species["species_id"].to_dict()   # indice de sortie -> species_id

# Contrôle : d'après la page Zenodo des modèles, la sortie 2 correspond à l'espèce 1355870.
assert class_map[2] == 1355870, "species_ids.csv ne suit pas l'ordre des sorties du modèle (class_mapping.txt n'est pas dans les entrées)"


In [ ]:
device = torch.device("cuda")
model = timm.create_model(
    "vit_base_patch14_reg4_dinov2.lvd142m",
    pretrained=False,                 # pas de téléchargement : les poids viennent de MODEL_CKPT
    num_classes=len(df_species),
    checkpoint_path=MODEL_CKPT,
)
model = model.to(device).eval()
data_config = timm.data.resolve_model_data_config(model)


In [ ]:
SCORE_MIN = 0.1  # probabilité softmax minimale pour retenir une espèce

_RESAMPLE = {
    "nearest": Image.Resampling.NEAREST,
    "bilinear": Image.Resampling.BILINEAR,
    "bicubic": Image.Resampling.BICUBIC,
}


def predict_species(quadrat_id: str) -> list[int]:
    """Renvoie les espèces retenues pour un quadrat du test. Le modèle est déjà chargé."""
    path = image_path(quadrat_id)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    image = Image.open(path).convert("RGB")
    _, height, width = data_config["input_size"]
    image = image.resize((width, height), _RESAMPLE[data_config["interpolation"]])

    pixels = np.asarray(image, dtype=np.float32) / 255.0
    mean = np.asarray(data_config["mean"], dtype=np.float32)
    std = np.asarray(data_config["std"], dtype=np.float32)
    pixels = (pixels - mean) / std

    batch = torch.from_numpy(np.ascontiguousarray(pixels)).permute(2, 0, 1).unsqueeze(0)
    batch = batch.to(device)

    with torch.no_grad(), torch.autocast("cuda"):
        scores = torch.softmax(model(batch).float(), dim=1)[0]

    chosen = torch.nonzero(scores >= SCORE_MIN, as_tuple=False).flatten()
    if chosen.numel() == 0:
        return []

    chosen = chosen[torch.argsort(scores[chosen], descending=True)]
    species = []
    seen = set()
    for index in chosen.tolist():
        species_id = int(class_map[int(index)])
        if species_id not in seen:
            seen.add(species_id)
            species.append(species_id)
    return species


In [ ]:
def create_submission_file(out_path: str = "/kaggle/working/submission.csv", predict=None) -> pd.DataFrame:
    """Parcourt PlantCLEF2025_test.csv dans l'ordre, appelle predict_species, écrit le CSV."""
    if predict is None:
        predict = predict_species

    ids = pd.read_csv(TEST_CSV, sep=";")[ID_COL].astype(str).tolist()   # ce CSV est séparé par des ;

    rows = []
    for qid in tqdm(ids):
        species = list(dict.fromkeys(int(s) for s in predict(qid)))   # int Python + sans doublons
        rows.append(str(species))

    df_sub = pd.DataFrame({"quadrat_id": ids, "species_ids": rows})
    df_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    return df_sub


In [ ]:
df_sub = create_submission_file()
print(df_sub.shape)
print(df_sub.head())
print("quadrats sans espèce :", (df_sub["species_ids"] == "[]").sum())
